In [1]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path.home() / "marian-nmt"
MARIAN_DIR = PROJECT / "marian" / "build"
MARIAN = MARIAN_DIR / "marian"

print("Python:", sys.version)
print("Project:", PROJECT)
print("Marian directory:", MARIAN_DIR)

result = subprocess.run(
    [str(MARIAN), "--version"],
    capture_output=True,
    text=True
)

print("Marian:", (result.stdout + result.stderr).strip())

Python: 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
Project: /home/suye/marian-nmt
Marian directory: /home/suye/marian-nmt/marian/build
Marian: v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800


# Neural Machine Translation: Myanmar ↔ Hieroglyphic

## 1. Objective

This experiment investigates neural machine translation between Myanmar
and Hieroglyphic text using the Marian NMT framework.

Two neural architectures are evaluated:

1. Sequence-to-Sequence
2. Transformer

Both translation directions are evaluated:

- Myanmar → Hieroglyphic
- Hieroglyphic → Myanmar

Hyperparameters are subsequently tuned to investigate their effect on
translation performance.

## 2. Environment

The experiments are conducted using Marian NMT in a WSL2 Ubuntu
environment. Training is performed on CPU.

## 3. Dataset

The experiment uses a parallel Myanmar–Hieroglyphic corpus consisting of
8,500 training pairs, 1,000 validation pairs, and 500 test pairs.

The corpus is already whitespace-segmented.

In [8]:
DATA = PROJECT / "data" / "myhi"

for split in ["train", "valid", "test"]:
    my_file = DATA / f"{split}.my"
    hi_file = DATA / f"{split}.hi"

    my_count = sum(1 for _ in open(my_file, encoding="utf-8"))
    hi_count = sum(1 for _ in open(hi_file, encoding="utf-8"))

    print(
        f"{split:5s}: "
        f"Myanmar={my_count:,}, "
        f"Hiero={hi_count:,}"
    )

train: Myanmar=8,500, Hiero=8,500
valid: Myanmar=1,000, Hiero=1,000
test : Myanmar=500, Hiero=500


In [9]:
my_lines = (DATA / "train.my").read_text(encoding="utf-8").splitlines()
hi_lines = (DATA / "train.hi").read_text(encoding="utf-8").splitlines()

for i in range(5):
    print(f"Example {i + 1}")
    print("Myanmar:", my_lines[i])
    print("Hiero:  ", hi_lines[i])
    print()

Example 1
Myanmar: မင်းက အေးဆေးတယ်
Hiero:   𓂨𓀀𓎡𓋴𓂝 𓏮𓀀𓐍𓂋

Example 2
Myanmar: ရိုးသားမှုရှိတယ်
Hiero:   𓏏𓂝𓋴𓂝 𓈙𓆑𓎛𓏭𓂝𓋴𓂝

Example 3
Myanmar: ကျွန်မ ရိုးသားနိုင်လား။
Hiero:   𓏭𓄿𓎛 𓆑 𓇋𓂝 𓊪𓐍𓎛𓂝𓈙𓈎

Example 4
Myanmar: ဘယ်လို သက်သာမှုလဲ။
Hiero:   𓏏𓊪𓄿𓈎 𓄿 𓋴𓂝𓂋𓆑𓂝𓅱

Example 5
Myanmar: ကျွန်မသွားခွင့်ပြုပါ။
Hiero:   𓄿𓂋𓂋𓐍𓏏 𓉔𓂝 𓈎𓐍 𓃀𓐍



In [10]:
def max_tokens(path):
    lines = path.read_text(encoding="utf-8").splitlines()
    return max(len(line.split()) for line in lines)

print("Maximum Myanmar tokens:",
      max_tokens(DATA / "train.my"))

print("Maximum Hiero tokens:",
      max_tokens(DATA / "train.hi"))

Maximum Myanmar tokens: 6
Maximum Hiero tokens: 5


## 4. Data Preprocessing

The parallel corpus is already whitespace-segmented. Separate source and
target vocabularies are therefore constructed from the training data using
Marian's `marian-vocab` utility.

The vocabulary is generated only from the training set to avoid introducing
information from the validation or test sets.

In [11]:
VOCAB_DIR = DATA / "vocab"
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

print("Vocabulary directory:", VOCAB_DIR)

Vocabulary directory: /home/suye/marian-nmt/data/myhi/vocab


In [12]:
my_vocab = VOCAB_DIR / "vocab.my.yml"

result = subprocess.run(
    [
        str(MARIAN_DIR / "marian-vocab"),
    ],
    stdin=open(DATA / "train.my", encoding="utf-8"),
    stdout=open(my_vocab, "w", encoding="utf-8"),
    stderr=subprocess.PIPE,
    text=True,
)

print("Myanmar vocabulary created:", my_vocab)
print("Size:", my_vocab.stat().st_size, "bytes")

if result.stderr:
    print(result.stderr)

Myanmar vocabulary created: /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml
Size: 319891 bytes
[2026-10-02 09:50:18] Creating vocabulary...
[2026-10-02 09:50:18] [data] Creating vocabulary stdout from stdin
[2026-10-02 09:50:18] Finished



In [13]:
hi_vocab = VOCAB_DIR / "vocab.hi.yml"

result = subprocess.run(
    [
        str(MARIAN_DIR / "marian-vocab"),
    ],
    stdin=open(DATA / "train.hi", encoding="utf-8"),
    stdout=open(hi_vocab, "w", encoding="utf-8"),
    stderr=subprocess.PIPE,
    text=True,
)

print("Hiero vocabulary created:", hi_vocab)
print("Size:", hi_vocab.stat().st_size, "bytes")

if result.stderr:
    print(result.stderr)

Hiero vocabulary created: /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml
Size: 86470 bytes
[2026-10-02 09:50:20] Creating vocabulary...
[2026-10-02 09:50:20] [data] Creating vocabulary stdout from stdin
[2026-10-02 09:50:20] Finished



In [14]:
print("Myanmar vocab:", my_vocab.exists())
print("Hiero vocab:", hi_vocab.exists())

Myanmar vocab: True
Hiero vocab: True


# 5. Experiments 
## 5.1 Experiment 1 — Myanmar → Hieroglyphic Seq2Seq

The first experiment uses a recurrent sequence-to-sequence encoder-decoder
architecture for Myanmar-to-Hieroglyphic translation.

This model serves as the first baseline for comparison with the Transformer
architecture.

In [15]:
MY_HI_S2S = PROJECT / "experiments" / "my-hi-seq2seq"

MODEL_DIR = MY_HI_S2S / "model"
OUTPUT_DIR = MY_HI_S2S / "output"
LOG_DIR = MY_HI_S2S / "logs"

for directory in [MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(MY_HI_S2S)

/home/suye/marian-nmt/experiments/my-hi-seq2seq


### Seq2Seq baseline configuration

A small recurrent encoder-decoder model is used as the baseline.
The model size is intentionally limited because training is performed
on CPU with approximately 7.6 GB of RAM available to WSL.

In [16]:
s2s_config = MY_HI_S2S / "config.yml"

s2s_config.write_text(
f"""type: s2s

model: {MODEL_DIR / "model.npz"}

train-sets:
  - {DATA / "train.my"}
  - {DATA / "train.hi"}

vocabs:
  - {my_vocab}
  - {hi_vocab}

valid-sets:
  - {DATA / "valid.my"}
  - {DATA / "valid.hi"}

dim-emb: 256
dim-rnn: 256

enc-type: bidirectional
enc-cell: gru
enc-depth: 1

dec-cell: gru
dec-depth: 1

dropout-rnn: 0.2
dropout-src: 0.1
dropout-trg: 0.1

mini-batch: 32
learn-rate: 0.0005

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: {LOG_DIR / "train.log"}
valid-log: {LOG_DIR / "valid.log"}

overwrite: true
""",
encoding="utf-8"
)

print(s2s_config.read_text())

type: s2s

model: /home/suye/marian-nmt/experiments/my-hi-seq2seq/model/model.npz

train-sets:
  - /home/suye/marian-nmt/data/myhi/train.my
  - /home/suye/marian-nmt/data/myhi/train.hi

vocabs:
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml

valid-sets:
  - /home/suye/marian-nmt/data/myhi/valid.my
  - /home/suye/marian-nmt/data/myhi/valid.hi

dim-emb: 256
dim-rnn: 256

enc-type: bidirectional
enc-cell: gru
enc-depth: 1

dec-cell: gru
dec-depth: 1

dropout-rnn: 0.2
dropout-src: 0.1
dropout-trg: 0.1

mini-batch: 32
learn-rate: 0.0005

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: /home/suye/marian-nmt/experiments/my-hi-seq2seq/logs/train.log
valid-log: /home/suye/marian-nmt/experiments/my-hi-seq2seq/logs/valid.log

overwrite: true



In [17]:
print(s2s_config)
print()
print(s2s_config.read_text())

/home/suye/marian-nmt/experiments/my-hi-seq2seq/config.yml

type: s2s

model: /home/suye/marian-nmt/experiments/my-hi-seq2seq/model/model.npz

train-sets:
  - /home/suye/marian-nmt/data/myhi/train.my
  - /home/suye/marian-nmt/data/myhi/train.hi

vocabs:
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml

valid-sets:
  - /home/suye/marian-nmt/data/myhi/valid.my
  - /home/suye/marian-nmt/data/myhi/valid.hi

dim-emb: 256
dim-rnn: 256

enc-type: bidirectional
enc-cell: gru
enc-depth: 1

dec-cell: gru
dec-depth: 1

dropout-rnn: 0.2
dropout-src: 0.1
dropout-trg: 0.1

mini-batch: 32
learn-rate: 0.0005

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: /home/suye/marian-nmt/experiments/my-hi-seq2seq/logs/train.log
valid-log: /home/suye/marian-nmt/experiments/my-hi-seq2seq/logs/valid.log

overwrite: true



In [18]:
train_cmd = [
    str(MARIAN),
    "-c",
    str(s2s_config),
    "--workspace",
    "2000",
]

print("Running:")
print(" ".join(train_cmd))

result = subprocess.run(
    train_cmd,
    cwd=MY_HI_S2S,
    text=True,
)

print("\nTraining process finished with return code:", result.returncode)

Running:
/home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/my-hi-seq2seq/config.yml --workspace 2000


[2026-10-02 09:50:33] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-10-02 09:50:33] [marian] Running on LAPTOP-8KUI1O0I as process 1416 with command line:
[2026-10-02 09:50:33] [marian] /home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/my-hi-seq2seq/config.yml --workspace 2000
[2026-10-02 09:50:33] [config] after: 0e
[2026-10-02 09:50:33] [config] after-batches: 0
[2026-10-02 09:50:33] [config] after-epochs: 0
[2026-10-02 09:50:33] [config] all-caps-every: 0
[2026-10-02 09:50:33] [config] allow-unk: false
[2026-10-02 09:50:33] [config] authors: false
[2026-10-02 09:50:33] [config] beam-size: 12
[2026-10-02 09:50:33] [config] bert-class-symbol: "[CLS]"
[2026-10-02 09:50:33] [config] bert-mask-symbol: "[MASK]"
[2026-10-02 09:50:33] [config] bert-masking-fraction: 0.15
[2026-10-02 09:50:33] [config] bert-sep-symbol: "[SEP]"
[2026-10-02 09:50:33] [config] bert-train-type-embeddings: true
[2026-10-02 09:50:33] [config] bert-type-vocab-size: 2



Training process finished with return code: 0


## 5.1 Experiment 2 — Myanmar → Hieroglyphic Transformer

The second experiment uses a transformer architecture for Myanmar-to-Hieroglyphic translation.

In [17]:
MY_HI_TRANSFORMER = PROJECT / "experiments" / "my-hi-transformer"

MODEL_DIR = MY_HI_TRANSFORMER / "model"
OUTPUT_DIR = MY_HI_TRANSFORMER / "output"
LOG_DIR = MY_HI_TRANSFORMER / "logs"

for directory in [MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", MY_HI_TRANSFORMER)

Experiment directory: /home/suye/marian-nmt/experiments/my-hi-transformer


In [20]:
transformer_config = MY_HI_TRANSFORMER / "config.yml"

transformer_config.write_text(
f"""type: transformer

model: {MODEL_DIR / "model.npz"}

train-sets:
  - {DATA / "train.my"}
  - {DATA / "train.hi"}

vocabs:
  - {my_vocab}
  - {hi_vocab}

valid-sets:
  - {DATA / "valid.my"}
  - {DATA / "valid.hi"}

dim-emb: 256
dim-rnn: 256

enc-depth: 2
dec-depth: 2

transformer-heads: 4
transformer-postprocess: dan
transformer-postprocess-emb: d
transformer-dropout: 0.1
transformer-dropout-attention: 0.1
transformer-dropout-ffn: 0.1

mini-batch: 16
learn-rate: 0.0003

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: {LOG_DIR / "train.log"}
valid-log: {LOG_DIR / "valid.log"}

overwrite: true
""",
encoding="utf-8"
)

print(transformer_config.read_text())

type: transformer

model: /home/suye/marian-nmt/experiments/my-hi-transformer/model/model.npz

train-sets:
  - /home/suye/marian-nmt/data/myhi/train.my
  - /home/suye/marian-nmt/data/myhi/train.hi

vocabs:
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml

valid-sets:
  - /home/suye/marian-nmt/data/myhi/valid.my
  - /home/suye/marian-nmt/data/myhi/valid.hi

dim-emb: 256
dim-rnn: 256

enc-depth: 2
dec-depth: 2

transformer-heads: 4
transformer-postprocess: dan
transformer-postprocess-emb: d
transformer-dropout: 0.1
transformer-dropout-attention: 0.1
transformer-dropout-ffn: 0.1

mini-batch: 16
learn-rate: 0.0003

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: /home/suye/marian-nmt/experiments/my-hi-transformer/logs/train.log
valid-log: /home/suye/marian-nmt/experiments/my-hi-transformer/logs/valid.log

overwrite: true

In [21]:
print(transformer_config.exists())
print(transformer_config)

True
/home/suye/marian-nmt/experiments/my-hi-transformer/config.yml


In [22]:
train_cmd = [
    str(MARIAN),
    "-c",
    str(transformer_config),
    "--workspace",
    "2000",
]

print("Running:")
print(" ".join(train_cmd))

result = subprocess.run(
    train_cmd,
    cwd=MY_HI_TRANSFORMER,
    text=True,
)

print("\nTraining process finished with return code:", result.returncode)

Running:
/home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/my-hi-transformer/config.yml --workspace 2000


[2026-10-01 13:25:15] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-10-01 13:25:15] [marian] Running on LAPTOP-8KUI1O0I as process 19164 with command line:
[2026-10-01 13:25:15] [marian] /home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/my-hi-transformer/config.yml --workspace 2000
[2026-10-01 13:25:15] [config] after: 0e
[2026-10-01 13:25:15] [config] after-batches: 0
[2026-10-01 13:25:15] [config] after-epochs: 0
[2026-10-01 13:25:15] [config] all-caps-every: 0
[2026-10-01 13:25:15] [config] allow-unk: false
[2026-10-01 13:25:15] [config] authors: false
[2026-10-01 13:25:15] [config] beam-size: 12
[2026-10-01 13:25:15] [config] bert-class-symbol: "[CLS]"
[2026-10-01 13:25:15] [config] bert-mask-symbol: "[MASK]"
[2026-10-01 13:25:15] [config] bert-masking-fraction: 0.15
[2026-10-01 13:25:15] [config] bert-sep-symbol: "[SEP]"
[2026-10-01 13:25:15] [config] bert-train-type-embeddings: true
[2026-10-01 13:25:15] [config] bert-type-vocab-siz


Training process finished with return code: 0


## 5.3 Experiment 3 — Hieroglyphic → Myanmar Seq2Seq

The third experiment uses a recurrent sequence-to-sequence encoder-decoder
architecture for Hieroglyphic-to-Myanmar translation.

In [23]:
HI_MY_S2S = PROJECT / "experiments" / "hi-my-seq2seq"

MODEL_DIR = HI_MY_S2S / "model"
OUTPUT_DIR = HI_MY_S2S / "output"
LOG_DIR = HI_MY_S2S / "logs"

for directory in [MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", HI_MY_S2S)

Experiment directory: /home/suye/marian-nmt/experiments/hi-my-seq2seq


In [24]:
hi_my_s2s_config = HI_MY_S2S / "config.yml"

hi_my_s2s_config.write_text(
f"""type: s2s

model: {MODEL_DIR / "model.npz"}

train-sets:
  - {DATA / "train.hi"}
  - {DATA / "train.my"}

vocabs:
  - {hi_vocab}
  - {my_vocab}

valid-sets:
  - {DATA / "valid.hi"}
  - {DATA / "valid.my"}

dim-emb: 256
dim-rnn: 256

enc-type: bidirectional
enc-cell: gru
enc-depth: 1

dec-cell: gru
dec-depth: 1

dropout-rnn: 0.2
dropout-src: 0.1
dropout-trg: 0.1

mini-batch: 32
learn-rate: 0.0005

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: {LOG_DIR / "train.log"}
valid-log: {LOG_DIR / "valid.log"}

overwrite: true
""",
encoding="utf-8"
)

print(hi_my_s2s_config.read_text())

type: s2s

model: /home/suye/marian-nmt/experiments/hi-my-seq2seq/model/model.npz

train-sets:
  - /home/suye/marian-nmt/data/myhi/train.hi
  - /home/suye/marian-nmt/data/myhi/train.my

vocabs:
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml

valid-sets:
  - /home/suye/marian-nmt/data/myhi/valid.hi
  - /home/suye/marian-nmt/data/myhi/valid.my

dim-emb: 256
dim-rnn: 256

enc-type: bidirectional
enc-cell: gru
enc-depth: 1

dec-cell: gru
dec-depth: 1

dropout-rnn: 0.2
dropout-src: 0.1
dropout-trg: 0.1

mini-batch: 32
learn-rate: 0.0005

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: /home/suye/marian-nmt/experiments/hi-my-seq2seq/logs/train.log
valid-log: /home/suye/marian-nmt/experiments/hi-my-seq2seq/logs/valid.log

overwrite: true



In [25]:
print("SOURCE:")
print(DATA / "train.hi")
print("VOCAB:")
print(hi_vocab)

print("\nTARGET:")
print(DATA / "train.my")
print("VOCAB:")
print(my_vocab)

SOURCE:
/home/suye/marian-nmt/data/myhi/train.hi
VOCAB:
/home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml

TARGET:
/home/suye/marian-nmt/data/myhi/train.my
VOCAB:
/home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml


In [26]:
train_cmd = [
    str(MARIAN),
    "-c",
    str(hi_my_s2s_config),
    "--workspace",
    "2000",
]

print("Running:")
print(" ".join(train_cmd))

result = subprocess.run(
    train_cmd,
    cwd=HI_MY_S2S,
    text=True,
)

print("\nTraining process finished with return code:", result.returncode)

Running:
/home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/hi-my-seq2seq/config.yml --workspace 2000


[2026-10-01 13:44:12] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-10-01 13:44:12] [marian] Running on LAPTOP-8KUI1O0I as process 22532 with command line:
[2026-10-01 13:44:12] [marian] /home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/hi-my-seq2seq/config.yml --workspace 2000
[2026-10-01 13:44:12] [config] after: 0e
[2026-10-01 13:44:12] [config] after-batches: 0
[2026-10-01 13:44:12] [config] after-epochs: 0
[2026-10-01 13:44:12] [config] all-caps-every: 0
[2026-10-01 13:44:12] [config] allow-unk: false
[2026-10-01 13:44:12] [config] authors: false
[2026-10-01 13:44:12] [config] beam-size: 12
[2026-10-01 13:44:12] [config] bert-class-symbol: "[CLS]"
[2026-10-01 13:44:12] [config] bert-mask-symbol: "[MASK]"
[2026-10-01 13:44:12] [config] bert-masking-fraction: 0.15
[2026-10-01 13:44:12] [config] bert-sep-symbol: "[SEP]"
[2026-10-01 13:44:12] [config] bert-train-type-embeddings: true
[2026-10-01 13:44:12] [config] bert-type-vocab-size: 2


Training process finished with return code: 0


## 5.4 Experiment 4 — Hieroglyphic to Myanmar Transformer

The fourth required experiment uses a Transformer architecture for
Hieroglyphic-to-Myanmar translation.

In [19]:
HI_MY_TRANSFORMER = PROJECT / "experiments" / "hi-my-transformer"

MODEL_DIR = HI_MY_TRANSFORMER / "model"

OUTPUT_DIR = HI_MY_TRANSFORMER / "output"

LOG_DIR = HI_MY_TRANSFORMER / "logs"

for directory in [MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", HI_MY_TRANSFORMER)

Experiment directory: /home/suye/marian-nmt/experiments/hi-my-transformer


In [20]:
config_path = HI_MY_TRANSFORMER / "config.yml"

config_text = f"""
type: transformer

model: {MODEL_DIR}/model.npz

train-sets:
  - {DATA}/train.hi
  - {DATA}/train.my

vocabs:
  - {DATA}/vocab/vocab.hi.yml
  - {DATA}/vocab/vocab.my.yml

valid-sets:
  - {DATA}/valid.hi
  - {DATA}/valid.my

dim-emb: 256
dim-rnn: 256

enc-depth: 2
dec-depth: 2

transformer-heads: 4
transformer-postprocess: dan
transformer-postprocess-emb: d

transformer-dropout: 0.1
transformer-dropout-attention: 0.1
transformer-dropout-ffn: 0.1

mini-batch: 16
learn-rate: 0.0003

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: {HI_MY_TRANSFORMER}/train.log
valid-log: {HI_MY_TRANSFORMER}/valid.log

overwrite: true
"""

config_path.write_text(config_text.strip() + "\n", encoding="utf-8")

print(config_path)
print("\nConfiguration:\n")
print(config_text)

/home/suye/marian-nmt/experiments/hi-my-transformer/config.yml

Configuration:


type: transformer

model: /home/suye/marian-nmt/experiments/hi-my-transformer/model/model.npz

train-sets:
  - /home/suye/marian-nmt/data/myhi/train.hi
  - /home/suye/marian-nmt/data/myhi/train.my

vocabs:
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.hi.yml
  - /home/suye/marian-nmt/data/myhi/vocab/vocab.my.yml

valid-sets:
  - /home/suye/marian-nmt/data/myhi/valid.hi
  - /home/suye/marian-nmt/data/myhi/valid.my

dim-emb: 256
dim-rnn: 256

enc-depth: 2
dec-depth: 2

transformer-heads: 4
transformer-postprocess: dan
transformer-postprocess-emb: d

transformer-dropout: 0.1
transformer-dropout-attention: 0.1
transformer-dropout-ffn: 0.1

mini-batch: 16
learn-rate: 0.0003

max-length: 50

valid-freq: 500u
save-freq: 500u
disp-freq: 100u

valid-metrics:
  - ce-mean-words
  - bleu

early-stopping: 5
keep-best: true

seed: 42

log: /home/suye/marian-nmt/experiments/hi-my-transformer/train.log
valid-log: /home/

In [21]:
train_cmd = [
    str(MARIAN),
    "-c",
    str(config_path),
    "--workspace",
    "2000",
]

print("Starting training...")
print("Command:")
print(" ".join(train_cmd))

result = subprocess.run(
    train_cmd,
    cwd=HI_MY_TRANSFORMER,
    text=True
)

print("\nTraining finished.")
print("Return code:", result.returncode)

Starting training...
Command:
/home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/hi-my-transformer/config.yml --workspace 2000


[2026-10-02 10:03:37] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-10-02 10:03:37] [marian] Running on LAPTOP-8KUI1O0I as process 3754 with command line:
[2026-10-02 10:03:37] [marian] /home/suye/marian-nmt/marian/build/marian -c /home/suye/marian-nmt/experiments/hi-my-transformer/config.yml --workspace 2000
[2026-10-02 10:03:37] [config] after: 0e
[2026-10-02 10:03:37] [config] after-batches: 0
[2026-10-02 10:03:37] [config] after-epochs: 0
[2026-10-02 10:03:37] [config] all-caps-every: 0
[2026-10-02 10:03:37] [config] allow-unk: false
[2026-10-02 10:03:37] [config] authors: false
[2026-10-02 10:03:37] [config] beam-size: 12
[2026-10-02 10:03:37] [config] bert-class-symbol: "[CLS]"
[2026-10-02 10:03:37] [config] bert-mask-symbol: "[MASK]"
[2026-10-02 10:03:37] [config] bert-masking-fraction: 0.15
[2026-10-02 10:03:37] [config] bert-sep-symbol: "[SEP]"
[2026-10-02 10:03:37] [config] bert-train-type-embeddings: true
[2026-10-02 10:03:37] [config] bert-type-vocab-size


Training finished.
Return code: 0


# 6. Results Summary

Four required Neural Machine Translation experiments were conducted using
the Marian framework:

1. Myanmar → Hiero using Sequence-to-Sequence
2. Myanmar → Hiero using Transformer
3. Hiero → Myanmar using Sequence-to-Sequence
4. Hiero → Myanmar using Transformer

All experiments used the same training, validation, and test corpus. The
models were trained using Marian's CPU implementation.

The best validation BLEU score observed during each experiment is reported
below.

In [24]:
import pandas as pd

results = pd.DataFrame([
    {
        "Direction": "Myanmar → Hiero",
        "Architecture": "Seq2Seq",
        "Best Validation BLEU": 17.9567,
        "Best Validation CE": 3.57903,
        "Training Status": "Completed"
    },
    {
        "Direction": "Myanmar → Hiero",
        "Architecture": "Transformer",
        "Best Validation BLEU": 13.4299,
        "Best Validation CE": 3.4913,
        "Training Status": "Completed"
    },
    {
        "Direction": "Hiero → Myanmar",
        "Architecture": "Seq2Seq",
        "Best Validation BLEU": 19.7291,
        "Best Validation CE": 4.72056,
        "Training Status": "Completed"
    },
    {
        "Direction": "Hiero → Myanmar",
        "Architecture": "Transformer",
        "Best Validation BLEU": 0.0000,
        "Best Validation CE": 4.27131,
        "Training Status": "Completed"
    }
])

results

,Direction,Architecture,Best Validation BLEU,Best Validation CE,Training Status
0,Myanmar → Hiero,Seq2Seq,17.9567,3.57903,Completed
1,Myanmar → Hiero,Transformer,13.4299,3.49130,Completed
2,Hiero → Myanmar,Seq2Seq,19.7291,4.72056,Completed
3,Hiero → Myanmar,Transformer,0.0000,4.27131,Completed
